## 🔹 Task 1: Conceptual Questions


# Why can’t we just use the delta rule for learning weights of hidden layers?

The delta rule cannot directly train hidden layers because hidden neurons do not have a known target output or direct error value. In the output layer, the network can easily calculate the error by comparing the predicted output with the correct answer. For example, if the correct output is 1 and the network predicts 0.2, the error can be directly calculated. This error is then used to update the weights using the delta rule.

However, hidden neurons do not produce the final answer. They only pass information to the next layer and help the network reach the final prediction. Because of this, we do not know what the “correct” output of a hidden neuron should be. Therefore, hidden layers cannot directly calculate an error term like (t−y), which is required in the delta rule.

To solve this problem, neural networks use backpropagation. Backpropagation takes the error from the output layer and sends it backward through the network. In this way, the network can estimate how much each hidden neuron contributed to the final error and update the hidden layer weights accordingly. This allows deep neural networks with multiple hidden layers to learn effectively.


# How far is training neural networks non-deterministic? How does randomness influence training speed and/or resulting model performance?

Training neural networks is partially non-deterministic because the training process contains several sources of randomness. This means that even if we use the same dataset, same model architecture, and same training code, the final model may still produce slightly different results each time it is trained.

One major source of randomness is random weight initialization. Before training begins, the network weights are assigned random values. Since training starts from different initial points, the network may follow different learning paths and reach different solutions. Another source of randomness is the random shuffling of training data and mini-batches. Neural networks learn step by step, so changing the order of the data changes the sequence of weight updates, which can affect the final model.

Techniques such as dropout also add randomness by randomly disabling some neurons during training. This forces the network to learn more robust features and helps reduce overfitting. In addition, GPU computations and floating-point operations can sometimes create very small numerical differences that grow during training.

Randomness affects both training speed and model performance. Good random initialization can help the network converge faster and achieve better accuracy, while poor initialization may slow down learning or lead to weaker results. Similarly, different mini-batch orders may lead the model toward different local minima. Sometimes randomness helps the model escape poor solutions and improves generalization performance.

Therefore, neural network training is not completely deterministic. Small random differences during training can lead to different learning behaviors, training times, and final accuracies, even when the same model is trained multiple times.


## 🔹 Task 2: Batch Gradient Descent


# MNIST Dataset

Implement an MLP for solving the mnist digits dataset (https://www.tensorflow.org/datasets/catalog/mnist).

Each sample has a 28x28 grayscale image of a single handwritten digit (0-9), and the corresponding expected output, again 0-9.


In [16]:
# Optionally install dependencies, if necessary
# conda install conda install tensorflow tensorflow-datasets 
# pip install tensorflow tensorflow_datasets

In [17]:
from datetime import datetime
import os

# If errors including tensorflow_datasets, try:
# os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
# better do the following command in your conda environment and restart vs code
# conda install -c conda-forge protobuf=3.20.3

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.layers import Input, Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Model
from tensorboard import program
from pathlib import Path

## Data Processing

The dataset is already loaded and split into a training and a testing set.

You can directly use it for a tensorflow / keras model.


In [18]:
# Enable GPU memory growth if GPUs are available
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Load MNIST dataset
ds_train: tf.data.Dataset
ds_test: tf.data.Dataset
ds_info: tfds.core.DatasetInfo
(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
    batch_size=32,
)

# Normalize pixel values from [0, 255] to [0.0, 1.0]
ds_train = ds_train.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, tf.cast(y, tf.int32)))
ds_test = ds_test.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, tf.cast(y, tf.int32)))

## 💡 Hint for the Input Layer (Flattening)

Since the images in the MNIST dataset are loaded as 3-dimensional tensors with the shape `(28, 28, 1)` (Height × Width × Channels), but an **MLP (Multilayer Perceptron)** can only process **1-dimensional vectors**, you need to reshape the data first.

You have two main approaches to achieve this:

### Option A: Using the Keras `Flatten` Layer (Recommended)

This integrates the transformation directly as the first layer of your model architecture:

```python
input_layer = Input(shape=(28, 28, 1))
x = Flatten()(input_layer)  # Converts (28, 28, 1) into a flat vector of 784 values
# Follow up with your Dense layers here...
```

### Option B: Manual Flattening via NumPy / TensorFlow

Instead of using a Keras layer, you can transform the data directly within the data pipeline. Since the dataset is provided as a `tf.data.Dataset`, you can apply a `.map()` function to flatten the images (similarly to how you would use `.reshape()` on raw NumPy arrays):

```python
# Flattening the images within the tf.data.Dataset pipeline
ds_train = ds_train.map(lambda x, y: (tf.reshape(x, [-1, 784]), y))
ds_test = ds_test.map(lambda x, y: (tf.reshape(x, [-1, 784]), y))

# Your MLP model can now take the flat vector directly:
input_layer = Input(shape=(784,))
# Follow up with your Dense layers here...
```

### Why is this necessary?

A classic MLP consists of Dense layers (fully connected layers).
Each neuron in a Dense layer requires its own individual weight for every single input feature (pixel).
The Problem: Dense layers do not natively understand 2D spatial structures (like rows and columns).
They expect a single, continuous list of features.
The Solution: "Flattening" collapses the $28 \times 28$ pixel matrix into a single vector of $784$ elements ($28 \times 28 = 784$).
This structural change is required before the network can begin learning the patterns of the handwritten digits.


## MLP

**TODO:** Train and evaluate an MLP on the MNIST dataset.

**TODO:** Try varying the hyperparameters/architecture of your MLP and compare both changes in the performance as well as the number of parameters in the MLP.
